# 02 - Engenharia de atributos

A mesma função `build_features()` é usada no treino e no **serviço de scoring em produção**
(arquivo único `src/features.py`) — elimina o clássico problema de *train/serve skew*.

Transformações:
1. **Limpeza** — removemos as 2 colunas de vazamento do target (Naive Bayes) e o id.
2. **Mapeamentos** — renda → ponto médio da faixa; escolaridade → ordem ordinal.
3. **Features derivadas** — utilização de limite, ticket médio, interação inatividade×contatos,
   limite por produto, queda de atividade ponderada.
4. **One-hot** de gênero, estado civil e categoria do cartão (ordem determinística).
5. **Imputação** por mediana calculada **apenas no treino** (evita vazamento) e persistida
   para o serviço reproduzir.


In [1]:
%matplotlib inline
import sys, pathlib, os
ROOT = pathlib.Path(os.getcwd())
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 110

import features as fe
import business as bz
import train as T

RAW = ROOT / "data" / "raw" / "BankChurners.csv"
ART = ROOT / "artifacts"
FIG = ROOT / "docs" / "figures"
FIG.mkdir(parents=True, exist_ok=True)


In [2]:
# Matriz de features final (com NaN antes da imputação)
raw = T.load_raw(RAW)
X, feature_columns = fe.build_features(raw)
print("Features:", len(feature_columns))
print("X shape:", X.shape)
X.head()


Features: 32
X shape: (10127, 32)


,Avg_Open_To_Buy,Avg_Utilization_Ratio,Contacts_Count_12_mon,Credit_Limit,Customer_Age,Dependent_count,Months_Inactive_12_mon,Months_on_book,Total_Amt_Chng_Q4_Q1,Total_Ct_Chng_Q4_Q1,...,Gender_F,Gender_M,Marital_Status_Divorced,Marital_Status_Married,Marital_Status_Single,Marital_Status_Unknown,Card_Category_Blue,Card_Category_Gold,Card_Category_Platinum,Card_Category_Silver
0,11914.0,0.061,3.0,12691.0,45.0,3.0,1.0,39.0,1.335,1.625,...,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0
1,7392.0,0.105,2.0,8256.0,49.0,5.0,1.0,44.0,1.541,3.714,...,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0
2,3418.0,0.000,0.0,3418.0,51.0,3.0,1.0,36.0,2.594,2.333,...,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0
3,796.0,0.760,1.0,3313.0,40.0,4.0,4.0,34.0,1.405,2.333,...,1.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0
4,4716.0,0.000,0.0,4716.0,40.0,3.0,1.0,21.0,2.175,2.500,...,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0


In [3]:
# Quem tem NaN e por quê (categorias "Unknown")
has_nan = X.isna().sum()
has_nan = has_nan[has_nan > 0]
print("Colunas com NaN:", has_nan.to_dict() if len(has_nan) else "nenhuma")

medians = fe.median_imputation_values(X)
X_imp = fe.impute_medians(X, medians)
print("\nNaN após imputação:", int(X_imp.isna().sum().sum()))
X_imp.describe().T[["mean", "std", "min", "25%", "50%", "75%", "max"]].round(3)


Colunas com NaN: {'education_ordinal': 1519, 'monthly_income': 1112}

NaN após imputação: 0


,mean,std,min,25%,50%,75%,max
Avg_Open_To_Buy,7469.140,9090.685,3.000,1324.500,3474.000,9859.000,34516.000
Avg_Utilization_Ratio,0.275,0.276,0.000,0.023,0.176,0.503,0.999
Contacts_Count_12_mon,2.455,1.106,0.000,2.000,2.000,3.000,6.000
Credit_Limit,8631.954,9088.777,1438.300,2555.000,4549.000,11067.500,34516.000
Customer_Age,46.326,8.017,26.000,41.000,46.000,52.000,73.000
Dependent_count,2.346,1.299,0.000,1.000,2.000,3.000,5.000
Months_Inactive_12_mon,2.341,1.011,0.000,2.000,2.000,3.000,6.000
Months_on_book,35.928,7.986,13.000,31.000,36.000,40.000,56.000
Total_Amt_Chng_Q4_Q1,0.760,0.219,0.000,0.631,0.736,0.859,3.397
Total_Ct_Chng_Q4_Q1,0.712,0.238,0.000,0.582,0.702,0.818,3.714


In [4]:
# As features derivadas separam bem os grupos? (média por desfecho)
df = raw.copy()
df[fe.TARGET_COLUMN] = df[fe.TARGET_COLUMN].astype(str).str.strip()
df["is_churn"] = (df[fe.TARGET_COLUMN] == fe.TARGET_POSITIVE).astype(int)
derivadas = ["revolving_balance_ratio", "avg_ticket", "trans_amt_per_contact",
             "inactive_contacts", "limit_per_product", "trans_decay_score"]
g = X.reset_index(drop=True)[derivadas].copy()
g["is_churn"] = df["is_churn"].values
gd = g.groupby("is_churn", observed=True).mean().T
gd.columns = ["ficou", "saiu"]
gd["delta_%"] = ((gd["saiu"] - gd["ficou"]) / gd["ficou"].abs() * 100).round(1)
gd.round(2)


,ficou,saiu,delta_%
revolving_balance_ratio,0.30,0.16,-45.2
avg_ticket,62.43,63.59,1.9
trans_amt_per_contact,1619.54,851.41,-47.4
inactive_contacts,7.63,10.71,40.4
limit_per_product,2096.36,2288.93,9.2
trans_decay_score,3573.10,2348.69,-34.3


In [5]:
from sklearn.model_selection import train_test_split

y = (df[fe.TARGET_COLUMN] == fe.TARGET_POSITIVE).astype(int).reset_index(drop=True)
Xr = X.reset_index(drop=True)
X_train, X_hold, y_train, y_hold = train_test_split(Xr, y, test_size=0.20, stratify=y, random_state=42)
medians_train = fe.median_imputation_values(X_train)
print("Mediana de imputation (treino) salva, ex.:", {k: round(v, 2) for k, v in medians_train.items() if k in {"monthly_income", "education_ordinal"}})


Mediana de imputation (treino) salva, ex.: {'education_ordinal': 2.0, 'monthly_income': 50000.0}


### Por que estas features?

- `inactive_contacts` captura o **estado de abandono**: meses parado × contatos — dois sinais que
  juntos indicam frustração.
- `trans_decay_score` pondera a **queda de atividade** (Q4 vs Q1) pelo volume — cai tanto em quem
  reduziu uso quanto em quem quase não usa.
- `avg_ticket`, `limit_per_product` e `revolving_balance_ratio` descrevem **intensidade e
  profundidade** do relacionamento com o produto, entram direto no cálculo de valor (CLV).
